# wildfire_train_eval_hf — GRPO training + OpenEnv eval + plots (Hugging Face GPU)

**Single notebook** for the full pipeline: set secrets, clone the repo, install deps, preflight, **GRPO training**, then **untrained and trained** `eval_policy_http.py` against the live Space, comparison plots, **GRPO** curves from `grpo_wildfire/log.jsonl`, judge-friendly exports, optional Hub upload, and a quick `overall_mean` readout.

**Hugging Face runtime:** set **`GITHUB_TOKEN`** and **`HF_TOKEN`** in *Settings → Secrets* (this Space or GPU Notebook). Default **`BASE`** is `/home/user/app/wildfire-env`.

**Cell order (run top to bottom; training can be skipped if you only re-run eval, but always run 1–2 first each session):**

| Step | What |
|:---:|:---|
| 1 | Secrets and `REPO` / `BASE` |
| 2 | `git clone` or `git pull --rebase`, `pip install -e .[train]`, `SPACE_URL`, `submission_artifacts` |
| 3 | OpenEnv + training stack preflight |
| 4 | GRPO `deadline_v2_a10g` (20 iters, resume-safe) |
| 5 | Untrained OpenEnv eval |
| 6 | Trained eval + `--plot-training` |
| 7 | Bar/scatter/delta charts (untrained vs trained) |
| 8 | Interactive plots from `log.jsonl` |
| 9 | `generate_training_plots` → `submission_artifacts/` |
| 10 | Display training PNGs + `training_summary.md` |
| 11 | (Optional) upload `grpo_wildfire/` to a model repo |
| 12 | Print `overall_mean` from both eval JSONs |

`grpo_wildfire/log.jsonl` and checkpoints are local artifacts (not always in git).

## `deadline_v2_a10g` (Cell 4)

Used for the hackathon submission: **20** GRPO iters, curriculum easy → medium → hard, `group_size=4`, `seeds_per_iter=2`, `micro_batch_size=2`, A10G-friendly. The submission run **stopped at iter 17**; training is **resume-safe** via `grpo_wildfire/latest/`, `optimizer_state.pt`, `resume_state.json`. Promote the desired adapter to `grpo_wildfire/final_adapter/` before the trained eval if needed.


In [ ]:
# Cell 1: env vars + constants (run first each session)
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN in the Hugging Face Space/Notebook secrets.")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the Hugging Face Space/Notebook secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/home/user/app/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens: GITHUB_TOKEN + HF_TOKEN")


In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "matplotlib", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl", "websockets",
    ],
    check=True,
)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH updated with NVIDIA libs")

os.makedirs(os.path.join(BASE, "submission_artifacts"), exist_ok=True)
SPACE_URL = "https://chunchunmaru-101-wildfire-env.hf.space"
print("Install complete. cwd:", os.getcwd(), "  Space:", SPACE_URL)


In [ ]:
# Cell 3: preflight for OpenEnv + training stack
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("Preflight passed")


In [ ]:
# Cell 4: GRPO training (resume-safe) — `deadline_v2_a10g`
import subprocess, sys, os

_HERE = os.path.dirname(os.path.abspath(globals().get("__file__", ".")))
_PROJECT_ROOT = os.path.abspath(os.path.join(_HERE, ".."))
if not os.path.isfile(os.path.join(_PROJECT_ROOT, "train_grpo.py")):
    _PROJECT_ROOT = os.path.expanduser("~/app/wildfire-env")

subprocess.run([
    sys.executable, "-c",
    f'''
import sys
sys.path.insert(0, {_PROJECT_ROOT!r})
from train_grpo import Config, train

train(Config(
    total_iterations=20,
    task_curriculum=(("easy", 0, 5), ("medium", 5, 12), ("hard", 12, 20)),
    group_size=4,
    seeds_per_iter=2,
    micro_batch_size=2,
    max_episode_steps=20,
    lora_dropout=0.0,
    warmup_iters=3,
    save_every=6,
    run_name="deadline_v2_a10g",
))
'''
], cwd=_PROJECT_ROOT, check=True)

print("deadline_v2_a10g done (or stopped early). See grpo_wildfire/log.jsonl, grpo_wildfire/latest, grpo_wildfire/final_adapter.")


## Evaluation, plots, and submission artifacts

> **OpenEnv transport (visible to judges):** Cells 5 and 6 below shell out to [`eval_policy_http.py`](../eval_policy_http.py), which is the actual policy runner. That script uses the official **OpenEnv `EnvClient` over a persistent WebSocket session** (`reset` / `step` on `/ws`) via [`wildfire_env/client.py`](../wildfire_env/client.py) — a typed `EnvClient[WildfireAction, WildfireObservation, State]` subclass. The wildfire-specific `POST /grader` endpoint is the only raw-HTTP call (it isn't part of OpenEnv).
>
> Why WebSocket and not raw `POST /reset`/`POST /step`? OpenEnv's HTTP transport (`HTTPEnvServer`) constructs a fresh `Environment` per request and tears it down — fine for one-shot probes, but every `step` would reset to `step=1`. WebSocket keeps a single `Environment` alive across `reset → step → step → ...`, which is what multi-turn RL rollouts require. See the docstring at the top of `eval_policy_http.py` for the full rationale.


In [ ]:
# Cell 5: untrained OpenEnv eval (WebSocket + /grader)
import subprocess, sys
subprocess.run([
    sys.executable, "eval_policy_http.py", "--untrained", "--base-url", SPACE_URL,
    "--seeds-per-task", "5", "--output", "submission_artifacts/eval_untrained.json",
], check=True, cwd=BASE)
print("Wrote submission_artifacts/eval_untrained.json")


In [ ]:
# Cell 6: trained eval; `--plot-training` also writes training_* if grpo_wildfire/log.jsonl exists
import subprocess, sys
subprocess.run([
    sys.executable, "eval_policy_http.py", "--base-url", SPACE_URL, "--seeds-per-task", "5",
    "--output", "submission_artifacts/eval_trained.json", "--plot-training",
], check=True, cwd=BASE)
print("Wrote eval_trained.json; training_* if log present")


In [ ]:
# Cell 7: base vs fine-tuned (eval JSONs — same seeds)
import json
from pathlib import Path
import matplotlib.pyplot as plt

unp = Path(BASE) / "submission_artifacts/eval_untrained.json"
trp = Path(BASE) / "submission_artifacts/eval_trained.json"
if not (unp.is_file() and trp.is_file()):
    raise FileNotFoundError("Run cells 5 and 6 first.")
du = json.loads(unp.read_text(encoding="utf-8"))
dt = json.loads(trp.read_text(encoding="utf-8"))
tasks = [t for t in ("easy", "medium", "hard") if t in du and t in dt]

def by_seed(b):
    return {int(e["seed"]): float(e["score"]) for e in b["episodes"]}

print("Base:", du.get("model_name"), "| trained adapter:", dt.get("adapter_path"))
ou, ot = float(du["overall_mean"]), float(dt["overall_mean"])
print(f"Overall  untrained={ou:.4f}  trained={ot:.4f}  Δ={ot-ou:+.4f}")
print("| task | n | U | T | Δ |")
for t in tasks:
    a, b = float(du[t]["mean"]), float(dt[t]["mean"])
    n = len(du[t]["episodes"])
    print(f"| {t} | {n} | {a:.4f} | {b:.4f} | {b-a:+.4f} |")

w = 0.35
x = list(range(len(tasks)))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([i - w/2 for i in x], [float(du[t]["mean"]) for t in tasks], w, label="base", color="tab:blue")
ax.bar([i + w/2 for i in x], [float(dt[t]["mean"]) for t in tasks], w, label="LoRA", color="tab:orange")
ax.set_xticks(x, tasks)
ax.set_ylabel("mean /grader")
ax.set_title("Untrained vs trained (showcase seeds)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
col = {"easy": "tab:green", "medium": "tab:purple", "hard": "tab:red"}
for t in tasks:
    su, st = by_seed(du[t]), by_seed(dt[t])
    for s in sorted(su.keys() & st.keys()):
        ax.scatter(su[s], st[s], c=col.get(t, "gray"), s=45, alpha=0.85, edgecolors="w", linewidths=0.4)
    ax.scatter([], [], c=col.get(t, "k"), s=45, label=t)
ax.plot([0,1],[0,1],"k--",alpha=0.35)
ax.set_xlabel("untrained"); ax.set_ylabel("trained")
ax.legend(title="task")
ax.set_xlim(0,1.02); ax.set_ylim(0,1.02); ax.set_box_aspect(1)
plt.tight_layout()
plt.show()

deltas = []
for t in tasks:
    su, st = by_seed(du[t]), by_seed(dt[t])
    for s in su.keys() & st.keys():
        deltas.append(st[s] - su[s])
if deltas:
    fig, ax = plt.subplots(figsize=(6,3.5))
    ax.hist(deltas, bins=8, color="steelblue", edgecolor="w")
    ax.axvline(0, color="k", ls="--", alpha=0.5)
    ax.set_xlabel("trained - untrained")
    ax.set_title(f"Δ (n={len(deltas)} episodes)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 8: GRPO training metrics from `grpo_wildfire/log.jsonl` (interactive matplotlib)
import json
from pathlib import Path
import matplotlib.pyplot as plt

LOG = Path(BASE) / "grpo_wildfire" / "log.jsonl"
if not LOG.is_file():
    print("No", LOG, "— run Cell 4 first or copy grpo_wildfire/ here.")
else:
    def _load(p):
        r = []
        for ln in p.read_text(encoding="utf-8").splitlines():
            ln = ln.strip()
            if not ln: continue
            o = json.loads(ln)
            if "iter" in o: r.append(o)
        r.sort(key=lambda x: int(x["iter"]))
        return r
    def _spans(recs):
        if not recs: return []
        c, a = str(recs[0]["task_id"]), int(recs[0]["iter"]); p = a
        out = []
        for r in recs[1:]:
            t, it = str(r["task_id"]), int(r["iter"])
            if t != c: out.append((c, a, p)); c, a = t, it
            p = it
        out.append((c, a, p))
        return out
    recs = _load(LOG)
    it = [int(r["iter"]) for r in recs]
    sp = _spans(recs)
    tc = {"easy": "#d9f99d", "medium": "#fde68a", "hard": "#fecaca"}
    def sh(ax):
        for tid, a, b in sp: ax.axvspan(a-0.45, b+0.45, alpha=0.35, color=tc.get(tid, "#e5e7eb"))
    fig, axes = plt.subplots(3,1,figsize=(10,7), sharex=True)
    for ax in axes: sh(ax)
    axes[0].plot(it, [r["mean_return"] for r in recs], "o-", color="#2563eb")
    axes[0].set_ylabel("mean_return")
    axes[1].plot(it, [r["mean_grader_score"] for r in recs], "o-", color="#16a34a")
    axes[1].set_ylabel("mean_grader")
    axes[2].plot(it, [r.get("action_parse_success_rate",0) for r in recs], "o-", color="#ea580c")
    axes[2].set_ylabel("parse success"); axes[2].set_xlabel("iter")
    plt.suptitle(str(LOG))
    plt.tight_layout()
    plt.show()
    fig, axl = plt.subplots(2,2,figsize=(10,5))
    z = [axl[0,0],axl[0,1],axl[1,0],axl[1,1]]
    for ax in z: sh(ax)
    z[0].plot(it, [r["policy_loss"] for r in recs], "o-", color="#dc2626"); z[0].set_ylabel("policy_loss")
    z[1].plot(it, [r["kl_divergence"] for r in recs], "o-", color="#7c3aed"); z[1].set_ylabel("kl")
    z[2].plot(it, [r["clip_fraction"] for r in recs], "o-", color="#ea580c"); z[2].set_ylabel("clip")
    z[3].plot(it, [r.get("rollout_seconds",0) for r in recs], "o-", label="rollout_s", color="#0ea5e9")
    z[3].plot(it, [r.get("update_seconds",0) for r in recs], "s--", label="update_s", color="#64748b")
    z[3].set_ylabel("s"); z[3].legend(fontsize=8)
    for ax in (z[2], z[3]): ax.set_xlabel("iter")
    plt.tight_layout()
    plt.show()
    print("Loaded", len(recs), "iters; spans", sp)


In [ ]:
# Cell 9: write judge-friendly PNGs/SVG + training_summary (same as `python plot_training_curves.py`)
from pathlib import Path
from plot_training_curves import generate_training_plots
_log = Path(BASE) / "grpo_wildfire" / "log.jsonl"
_out = Path(BASE) / "submission_artifacts"
_out.mkdir(parents=True, exist_ok=True)
if not _log.is_file():
    print("Skip export: no", _log)
else:
    generate_training_plots(_log, _out)
    print("Wrote training_* to submission_artifacts/")


In [ ]:
# Cell 10: show training PNGs + training_summary in the notebook
from pathlib import Path
from IPython.display import Image, display, Markdown
_arts = Path(BASE) / "submission_artifacts"
for n in ("training_reward_curve.png", "training_loss_curve.png"):
    p = _arts / n
    if p.is_file():
        display(Markdown("### " + p.name))
        display(Image(filename=str(p)))
    else:
        print("Missing", p)
_s = _arts / "training_summary.md"
if _s.is_file():
    display(Markdown("### training_summary.md"))
    display(Markdown(_s.read_text(encoding="utf-8")))
else:
    print("No training_summary.md")


In [ ]:
# Cell 11 (optional): upload `grpo_wildfire` to a HF model repo
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo
HF_REPO_ID = os.environ.get("HF_REPO_ID", "Chunchunmaru-101/wildfire-grpo-checkpoints")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in secrets to upload")
api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(Path(BASE) / "grpo_wildfire"),
    repo_id=HF_REPO_ID, repo_type="model",
    commit_message="Upload wildfire GRPO checkpoints",
    ignore_patterns=["*.pyc", "__pycache__/*"],
)
print("Uploaded to https://huggingface.co/" + HF_REPO_ID)


In [ ]:
# Cell 12: quick `overall_mean` from eval JSONs
import json, os
for name in ("eval_untrained.json", "eval_trained.json"):
    p = os.path.join(BASE, "submission_artifacts", name)
    with open(p, encoding="utf-8") as f:
        d = json.load(f)
    print(name, "overall_mean =", d.get("overall_mean"))
